# Step 1. Convert Raw EDF Data to HDF5

Convert EDF projection files acquired at the **ID16A nano-imaging beamline (ESRF)** into a single
consolidated HDF5 archive following the [Data Exchange](http://dxchange.readthedocs.io/) layout.

The beamline uses a **Fresnel zone-plate** setup: the zone plate focuses the beam onto the sample,
and the detector is placed downstream at distance `focustodetectordistance`.  
Multiple sample-to-detector distances `z1` are recorded in a single scan to provide holographic
diversity (multi-distance phase retrieval).

**Outputs written to the HDF5 file**
| Dataset | Description |
|---------|-------------|
| `/exchange/data{k}` | Raw projections at distance index `k` |
| `/exchange/data_white_start{k}` / `data_white_end{k}` | Flat fields before / after scan |
| `/exchange/data_dark{k}` | Dark fields |
| `/exchange/theta` | Rotation angles (degrees) extracted from EDF headers |
| `/exchange/shifts` | Pre-measured sample motion shifts from `correct.txt` |
| `/exchange/voxelsize` | Effective voxel sizes per distance |
| `/exchange/z1` | Sample-to-focus distances (m) |

In [ ]:
import h5py 
import dxchange
import os
import numpy as np
from holotomocupy.utils import *

## Parameters

Edit the paths and geometry below to match your dataset.

- **`z1_ids`** – indices (0-based) of the distances to use from the full 8-distance scan.  
  Using 4 distances (`[0,1,2,3]`) gives a good balance of phase diversity and memory.
- **`sx0`** – longitudinal position of the focal spot (m), used to shift raw `z1` values to actual
  sample-to-focus distances.
- **`detector_pixelsize`** – physical pixel pitch on the detector (m); multiplied by 2 because the
  detector is operated in 2×2 binning.
- **Derived quantities** (computed automatically):
  - `z2 = focustodetectordistance - z1` — sample to detector distance  
  - `magnifications = focustodetectordistance / z1` — geometric magnification per distance  
  - `distances = z1*z2/focustodetectordistance` — effective propagation distance for phase retrieval  
  - `voxelsize = detector_pixelsize / magnification` — object-space voxel size

In [ ]:
n = 2048
ntheta = 1800
detector_pixelsize = 3.03751328912064e-6
energy = 33.35
wavelength = 1.24e-09 / energy
focustodetectordistance = 1.28


sx0 = 1.286e-3
z1 = np.array([4.236, 4.3625, 4.8685, 5.9195]) * 1e-3 - sx0
ndist = len(z1)
z2 = focustodetectordistance - z1


distances = (z1 * z2) / focustodetectordistance
magnifications = focustodetectordistance / z1
norm_magnifications = magnifications / magnifications[0]
voxelsizes = np.abs(detector_pixelsize / magnifications)
voxelsize = voxelsizes[0]

path = '/data2/vnikitin/atomium/20240924/AtomiumS2/'
pfile = 'AtomiumS2_HT_007nm'
path_out = '/data2/vnikitin/atomium_rec/20240924/AtomiumS2/'
file_out = f'data.h5'



### Parse files and save everything to h5

Iterate over all projection angles and distances.  For each distance `k` the EDF files are read,
cropped to an `n × n` region centred on the detector, and written directly into the preallocated
HDF5 datasets.  Rotation angles are extracted from the `motor_pos` field in the EDF header.

> **Tip:** Run this step once per dataset; subsequent steps read only from the HDF5 file.

In [ ]:
def find_angle(fname):
    """Extract rotation angle from EDF header.

    Reads only the ASCII header (until the closing `}`) in binary mode —
    avoids loading the full image (~8 MB per projection).
    """
    with open(fname, 'rb') as f:
        header = b''
        while b'}' not in header:
            header += f.read(512)
    for line in header.decode('latin-1').split('\n'):
        if 'motor_pos' in line:
            return float(line.split()[3])

os.makedirs(path_out, exist_ok=True)

# --- Crop coordinates (same detector size for all distances) ---
# Read one reference image to determine the detector dimensions, then compute
# the centred n×n crop once and reuse it for every distance and projection.
dname0 = f'{path}/{pfile}_1_'
n0, n1 = dxchange.read_edf(f'{dname0}/ref0000_0000.edf')[0].shape
sty, endy = n0 // 2 - n // 2, n0 // 2 + n // 2
stx, endx = n1 // 2 - n // 2, n1 // 2 + n // 2

with h5py.File(f'{path_out}/{file_out}', 'w') as fid:
    data_ds   = [fid.create_dataset(f'/exchange/data{k}',             shape=(ntheta, n, n), dtype='uint16') for k in range(ndist)]
    white0_ds = [fid.create_dataset(f'/exchange/data_white_start{k}', shape=(20, n, n),     dtype='uint16') for k in range(ndist)]
    white1_ds = [fid.create_dataset(f'/exchange/data_white_end{k}',   shape=(20, n, n),     dtype='uint16') for k in range(ndist)]
    dark_ds   = [fid.create_dataset(f'/exchange/data_dark{k}',        shape=(20, n, n),     dtype='uint16') for k in range(ndist)]

    theta_ds  = fid.create_dataset('/exchange/theta',  shape=(ntheta, ndist), dtype='float32')
    shifts_ds = fid.create_dataset('/exchange/shifts', shape=(ntheta, ndist, 2), dtype='float32')
    attrs_ds  = fid.create_dataset('/exchange/attrs',  shape=(ntheta, ndist, 3), dtype='float32')

    fid.create_dataset('/exchange/voxelsize',             data=voxelsizes,                dtype='float32')
    fid.create_dataset('/exchange/z1',                    data=z1,                        dtype='float32')
    fid.create_dataset('/exchange/detector_pixelsize',    data=[detector_pixelsize],      dtype='float32')
    fid.create_dataset('/exchange/energy',                data=[energy],                  dtype='float32')
    fid.create_dataset('/exchange/focusdetectordistance', data=[focustodetectordistance], dtype='float32')

    # --- Rotation angles (same for all distances — read once from distance 0) ---
    # The rotation motor position is identical across all z1 planes, so reading
    # ntheta headers from distance 0 is sufficient.
    theta_vals = np.array(
        [find_angle(f'{dname0}/{pfile}_1_{id:04}.edf') for id in range(ntheta)],
        dtype='float32',
    )
    theta_ds[:] = theta_vals[:, None]   # broadcast to all ndist columns

    for k in range(ndist):
        dname = f'{path}/{pfile}_{k + 1}_'
        shifts_ds[:, k] = np.loadtxt(f'{dname}/correct.txt')[:ntheta]
        attrs_ds[:,  k] = np.loadtxt(f'{dname}/attributes.txt')[:ntheta, :3]

        for id in range(20):
            white0_ds[k][id] = dxchange.read_edf(f'{dname}/ref{id:04}_0000.edf')[0][sty:endy, stx:endx]
            white1_ds[k][id] = dxchange.read_edf(f'{dname}/ref{id:04}_{ntheta:04}.edf')[0][sty:endy, stx:endx]
            dark_ds[k][id]   = dxchange.read_edf(f'{dname}/darkend{id:04}.edf')[0][sty:endy, stx:endx]

        for id in range(ntheta):
            fname = f'{dname}/{pfile}_{k + 1}_{id:04}.edf'
            data_ds[k][id] = dxchange.read_edf(fname)[0][sty:endy, stx:endx]
            if id % 100 == 0:
                print(id, k, theta_vals[id], shifts_ds[id, k])